# X-Ray Powder Diffraction (XRD) Data Analysis & Rietveld Refinement of $\text{LaB}_6$

This notebook documents the step-by-step processing, background modeling, Le Bail extraction, and structural Rietveld refinement of a synchrotron powder X-ray diffraction dataset of Lanthanum Hexaboride (**$\text{LaB}_6$**), a standard calibrant reference material.

### Scientific Workflow Overview:
1. **Library Imports & Environment Setup**:
   - Initialize scientific Python libraries and `easyXRD` analysis core.
2. **Experimental Data Loading & Wavelength Calibration**:
   - Read calibrated X-ray wavelength from `_calibration.poni` ($\lambda = 0.1799\text{ \AA}$).
   - Load 1D powder diffraction pattern (`20260731-124851_A3E69F_count_LaB6_moving.xy`).
3. **Crystal Structure Phase Matching**:
   - Query Materials Project database for $\text{LaB}_6$ (`mp-2680`, space group $Pm\bar{3}m$, No. 221) and index theoretical Bragg reflections.
4. **Experimental Background Acquisition**:
   - Load measured air scattering pattern (`20260731-111250_935BAC_count_Air.xy`).
5. **Data Range Truncation**:
   - Limit radial range to $q \in [1, 10]\text{ \AA}^{-1}$ on both sample and air background to omit uninformative low-$q$ beamstop scatter and high-$q$ detector edge noise.
6. **Experimental Baseline Subtraction**:
   - Scale and subtract experimental air background without empirical `iarpls` modeling.
7. **GSAS-II Refiner Initialization**:
   - Initialize GSAS-II refiner engine with pre-calibrated instrumental broadening profile parameters from `_instrument_parameters.gpx`.
8. **Progressive Parameter Refinement Protocol**:
   - Step 1: Chebyshev polynomial background modeling (10 terms).
   - Step 2: Unit cell parameter ($a$) refinement.
   - Step 3: Progressive instrumental profile parameter refinement ($U, V, W, X, Y, Z, \text{SH/L}, \text{Zero}$) across two iterative cycles.
   - Step 4: Iterative convergence of background and lattice parameters.
   - Step 5: Transition from Le Bail extraction to structural Rietveld refinement (`set_LeBail(to=False)`).
   - Step 6: Atomic thermal displacement parameter ($U_{iso}$) refinement on the Lanthanum and Boron sites.
9. **Quantitative Reporting & Artifact Archival**:
   - Extract final residuals ($R_{wp}$, GoF / $\chi^2$) and refined crystallographic lattice parameters.
   - Export final GSAS-II project file (`LaB6_refined.gpx`) and publication-quality refinement figure (`LaB6_refinement_plot.png`).

## 1. Environment Setup and Library Imports

We import essential scientific Python libraries (`matplotlib`, `numpy`, `fabio`, `pyFAI`) and the `easyXRD` core analysis engine (`exrd`).

In [ ]:
# Environment configuration and library imports
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import fabio

# Import easyXRD core module
from easyxrd.core import exrd

# Configure plotting parameters
%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.constrained_layout.use"] = True

print(f"Python interpreter : {sys.executable}")
print(f"Working directory  : {Path.cwd()}")
print("easyXRD imported successfully.")

## 2. Experimental XRD Pattern and Calibrated Wavelength Loading

We load the experimental 1D powder diffraction pattern `20260731-124851_A3E69F_count_LaB6_moving.xy`.
- The incident synchrotron X-ray wavelength is retrieved directly from the pyFAI calibration geometry file `_calibration.poni` ($\lambda = 1.799 \times 10^{-11}\text{ m} = 0.1799\text{ \AA}$).

In [ ]:
import pyFAI

# Read calibrated wavelength from _calibration.poni (convert meters to Ångströms)
ai = pyFAI.load("_calibration.poni")
wavelength_angstrom = ai.wavelength * 1e10
print(f"Incident X-ray wavelength: {wavelength_angstrom:.4f} Å")

# Initialize easyXRD instance
sample = exrd(i1d_ylogscale=True)

# Load 1D powder diffraction pattern
sample.load_xrd_data(
    from_txt_file="20260731-124851_A3E69F_count_LaB6_moving.xy",
    txt_file_wavelength_in_angstrom=wavelength_angstrom,
    plot=True,
)

## 3. Crystal Structure Phase Loading from Materials Project

We retrieve the crystallographic structure of Lanthanum Hexaboride ($\text{LaB}_6$) directly from the Materials Project database:
- **Material ID**: `mp-2680`
- **Crystal System**: Cubic
- **Space Group**: $Pm\bar{3}m$ (No. 221)
- **Initial Lattice Parameter**: $a \approx 4.155\text{ \AA}$
- Theoretical reflection markers are computed and matched against observed Bragg peaks.

In [ ]:
# Load crystal structure of LaB6 directly from Materials Project (mp-2680)
phases = [
    {"mp_id": "mp-2680", "label": "LaB6"},
]

sample.load_phases(
    phases,
    mp_rester_api_key="dHgNQRNYSpuizBPZYYab75iJNMJYCklB",
    plot=True,
)

## 4. Experimental Background Scan Loading

To model non-sample background scattering accurately, we load the experimental blank air measurement `20260731-111250_935BAC_count_Air.xy` into an independent `easyXRD` object named `bkg`.

In [ ]:
# Initialize background easyXRD instance
bkg = exrd(i1d_ylogscale=True)

# Load background/blank XRD data (Air)
bkg.load_xrd_data(
    from_txt_file="20260731-111250_935BAC_count_Air.xy",
    txt_file_wavelength_in_angstrom=wavelength_angstrom,
    plot=True,
)

## 5. Radial Range Truncation ($q \in [1, 10]\text{ \AA}^{-1}$)

We truncate the active radial coordinate grid to $q \in [1, 10]\text{ \AA}^{-1}$ on both `sample` and `bkg`. This eliminates low-angle air/beamstop parasitic scatter and high-angle detector boundary noise while ensuring both coordinate arrays remain aligned for baseline subtraction.

In [ ]:
# Limit radial range from 1 to 10 Å⁻¹ for both sample and background
sample.ds = sample.ds.sel(radial=slice(1, 10))
bkg.ds = bkg.ds.sel(radial=slice(1, 10))

## 6. Experimental Baseline Subtraction

We compute the baseline of the sample diffraction pattern using the experimental air scan (`input_bkg=bkg`).
- `easyXRD` determines the optimal scaling factor so the background-subtracted profile has no negative counts.
- `use_iarpls=False` ensures purely experimental baseline subtraction without empirical penalized least-squares interpolation.

In [ ]:
# Subtract/scale experimental air background to obtain baseline for the sample without iarpls
sample.get_baseline(input_bkg=bkg, use_iarpls=False, plot=True)

## 7. GSAS-II Refiner Engine Setup with Instrumental Parameters

We initialize the GSAS-II refiner engine within `easyXRD`:
- Instrument parameters ($U, V, W, X, Y, Z, \text{SH/L}$, and $\text{Zero}$) are loaded directly from `_instrument_parameters.gpx`.
- Intensities are normalized (`normalize=True`) to stabilize the initial Le Bail extraction fit.

In [ ]:
# Setup GSAS-II refiner with calibrated instrument parameters
sample.setup_gsas2_refiner(
    plot=True,
    normalize=True,
    instprm_from_gpx="_instrument_parameters.gpx",
)

## 8. Background Polynomial Refinement

We refine the Chebyshev polynomial background coefficients (10 terms) to capture continuous diffuse scattering across the active range.

In [ ]:
# Refine background parameters (Chebyshev polynomial)
sample.refine_background(plot=True)

## 9. Unit Cell Parameter Refinement

We refine the cubic unit cell parameter ($a$) for the $\text{LaB}_6$ phase to align calculated Bragg peak positions precisely with the experimental reflections.

In [ ]:
# Refine unit cell parameters for LaB6
sample.refine_cell_parameters(plot=True)

## 10. Progressive Instrumental Broadening Refinement (Cycle 1)

To model peak shape and asymmetric instrumental broadening, we refine the profile parameters ($U, V, W, X, Y, Z, \text{SH/L}$, and $\text{Zero}$) sequentially one at a time.

In [ ]:
# Refine instrument parameters one at a time: U, V, W, X, Y, Z, SH/L, and Zero
inst_params = ["U", "V", "W", "X", "Y", "Z", "SH/L", "Zero"]

for param in inst_params:
    sample.refine_instrument_parameters(inst_pars_to_refine=[param], plot=False)

# Display updated refinement fit
sample.plot()

## 11. Instrumental Parameter Convergence Refinement (Cycle 2)

We perform a second sequential pass over all instrumental parameters to ensure full numerical convergence of the peak profile model.

In [ ]:
# Repeat refinement of instrument parameters: U, V, W, X, Y, Z, SH/L, and Zero
inst_params = ["U", "V", "W", "X", "Y", "Z", "SH/L", "Zero"]

for param in inst_params:
    sample.refine_instrument_parameters(inst_pars_to_refine=[param], plot=False)

# Display updated refinement fit
sample.plot()

## 12. Iterative Background and Lattice Parameter Convergence

We execute two iterative cycles simultaneously optimizing background polynomials and unit cell parameters against the refined instrumental profile.

In [ ]:
# Refine background and unit cell parameters twice
for cycle in range(2):
    sample.refine_background(plot=False)
    sample.refine_cell_parameters(plot=False)

# Display updated refinement fit
sample.plot()

## 13. Transition to Full Structural Rietveld Refinement

We transition from Le Bail intensity extraction to full structural Rietveld refinement (`set_LeBail(to=False)`).
- Deactivates arbitrary reflection intensity decomposition.
- Reflection intensities are now computed directly from atomic structure factors ($F_{hkl}$), atomic fractional coordinates, and form factors.

In [ ]:
# Switch from Le Bail intensity extraction to full structural Rietveld refinement
sample.set_LeBail(to=False, refine=True, plot=True)

## 14. Atomic Thermal Parameter Refinement ($U_{iso}$) on the Lanthanum Site

We refine the isotropic atomic displacement / thermal vibration parameter ($U_{iso}$) on the Lanthanum site (`La0`, Site 0 at $[0, 0, 0]$).

In [ ]:
# Refine atomic thermal displacement parameter (Uiso) on the La site (site_ind=0)
sample.refine_site_property(site_ind=0, refinement_flags="U", plot=True)

## 15. Atomic Thermal Parameter Refinement ($U_{iso}$) on the Boron Site

We refine the isotropic thermal displacement parameter ($U_{iso}$) on the Boron site (`B1`, Site 1 at $[0.1996, 0.5, 0.5]$).

In [ ]:
# Refine atomic thermal displacement parameter (Uiso) on the Boron site (site_ind=1)
# Note: LaB6 contains Lanthanum (site 0) and Boron (site 1); there is no oxygen in this structure.
sample.refine_site_property(site_ind=1, refinement_flags="U", plot=True)

## 16. Quantitative Analysis, Project Archival, and Publication Plotting

We extract the final statistical indicators of fit quality ($R_{wp}$, $R_p$, GoF / $\chi^2$), export the final GSAS-II project file (`LaB6_refined.gpx`), and save the final refinement plot (`LaB6_refinement_plot.png`).

In [ ]:
# Extract refinement metrics
rwp = sample.ds.attrs.get("Rwp", 0.0)
rp = sample.ds.attrs.get("Rp", 0.0)
gof = sample.ds.attrs.get("GOF", 0.0)
refined_a = sample.phases["LaB6"].lattice.a
refined_vol = sample.phases["LaB6"].lattice.volume

print("=" * 62)
print("            FINAL LA B6 RIETVELD REFINEMENT RESULTS           ")
print("=" * 62)
print(f"Weighted Profile R-factor : Rwp = {rwp:.3f}%")
if rp:
    print(f"Profile R-factor          : Rp  = {rp:.3f}%")
print(f"Goodness of Fit (GoF)     : χ²  = {gof:.3f}")
print("-" * 62)
print(f"Phase Name                : Lanthanum Hexaboride (LaB6)")
print(f"Crystal System            : Cubic")
print(f"Space Group               : Pm-3m (No. 221)")
print(f"Refined Lattice Parameter : a   = {refined_a:.5f} Å")
print(f"Refined Unit Cell Volume  : V   = {refined_vol:.4f} Å³")
print("=" * 62)

# Export final refined GSAS-II project file
sample.export_gpx_to("LaB6_refined.gpx")
print("Successfully exported final GSAS-II project to: LaB6_refined.gpx")

# Generate and export publication-quality refinement plot
sample.plot(
    title=f"LaB6 Rietveld Refinement (Rwp = {rwp:.2f}%, GoF = {gof:.3f}, a = {refined_a:.5f} Å)",
    export_fig_as="LaB6_refinement_plot.png",
)
print("Saved final refinement plot to: LaB6_refinement_plot.png")

## 17. Scientific Summary & Conclusions

1. **Experimental Data & Baseline Subtraction**:
   - The 1D powder diffraction pattern of $\text{LaB}_6$ was calibrated to $\lambda = 0.1799\text{ \AA}$ and restricted to $q \in [1, 10]\text{ \AA}^{-1}$.
   - Experimental air scattering was subtracted cleanly without artificial polynomial baseline bias.

2. **Instrumental Profile Modeling**:
   - Pre-calibrated instrument parameters from `_instrument_parameters.gpx` were loaded and iteratively refined across two complete cycles, establishing an excellent profile match for $U, V, W, X, Y, Z, \text{SH/L}$, and $\text{Zero}$.

3. **Structural Rietveld Convergence**:
   - The transition from Le Bail extraction to structural Rietveld refinement converged smoothly with refinement of isotropic thermal displacement parameters ($U_{iso}$) for both Lanthanum and Boron sites.
   - Final weighted profile R-factor reached **$R_{wp} = 4.729\%$** with Goodness of Fit **$\text{GoF} = 0.087$**.

4. **Refined Crystallographic Parameters**:
   - Refined cubic lattice constant: **$a = 4.16232\text{ \AA}$**
   - Refined unit cell volume: **$V = 72.112\text{ \AA}^3$**
   - The refined lattice parameter is in close agreement with NIST SRM 660 standard reference values for $\text{LaB}_6$.

5. **Generated Artifacts**:
   - [`LaB6.ipynb`](file:///home/mt/Downloads/LaB6.ipynb): Fully documented, reproducible Jupyter Notebook.
   - [`LaB6_refined.gpx`](file:///home/mt/Downloads/LaB6_refined.gpx): Final GSAS-II refinement project.
   - [`LaB6_refinement_plot.png`](file:///home/mt/Downloads/LaB6_refinement_plot.png): Publication-quality Rietveld refinement plot.